# LanceDB fundamentals

LanceDB is an open-source vector database designed to handle large-scale vector data efficiently. It provides a robust platform for storing, indexing, and querying high-dimensional vectors, making it ideal for applications in machine learning, computer vision, and natural language processing.

We'll be using LanceDB for the purpose of creating RAG applications. However before we dive into that, let's cover some fundamental concepts of LanceDB, so that you know what is happening under the hood.

## Connect LanceDB

- when connecting to LanceDB it will create a folder for your lancedb database if it does not already exist

In [2]:
import lancedb

db = lancedb.connect("knowledge_base")
db

LanceDBConnection(uri='c:\\Users\\indir\\Documents\\github\\pydantic_nvdia\\lancedb\\knowledge_base')

## Create a Table

when creating a table, it is stored as a folder with .lance suffix containing the following folders
- _transactions
- _versions
- data

a table is a pyarraow table stored in the data folder

In [3]:
import json
with open("data/animals_text_embeddings.json") as f:
    data = json.loads(f.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
data[0].get("text")

'A small brown dog running.'

In [5]:
data[0].get("vector")

[0.12, 0.85, 0.33]

In [6]:
db.create_table("animals_text", exist_ok=True, data=data)
db["animals_text"]


LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\indir\\Documents\\github\\pydantic_nvdia\\lancedb\\knowledge_base'))

In [7]:
# it's a pyarrow table 
db["animals_text"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [8]:
# converting to pandas 
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


add some more data to the table

In [9]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

db["animals_text"].add(more_data)

AddResult(version=3)

In [10]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


# Drop a table


In [11]:
db

LanceDBConnection(uri='c:\\Users\\indir\\Documents\\github\\pydantic_nvdia\\lancedb\\knowledge_base')

In [12]:
from lancedb.pydantic import LanceModel

class CoolSchema(LanceModel):
    cool_name: str
    cool_number: int = 42
db.create_table("cool_table", schema=CoolSchema, mode="overwrite")

# mode overwrite -> replace in SQL

LanceTable(name='cool_table', _conn=LanceDBConnection(uri='c:\\Users\\indir\\Documents\\github\\pydantic_nvdia\\lancedb\\knowledge_base'))

In [13]:
import pandas as pd

cool_data = [
    {"cool_name": "pi", "cool_number": 3.14},
    {"cool_name": "e",  "cool_number": 2.718},
]
cool_df = pd.DataFrame(cool_data)
cool_df

,cool_name,cool_number
0,pi,3.140
1,e,2.718


In [14]:
db["cool_table"].add(cool_df)

AddResult(version=2)

In [15]:
db.drop_table("cool_table")

In [16]:
db.table_names()

C:\Users\indir\AppData\Local\Temp\ipykernel_5708\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

## Open existing table

In [17]:
# since db is already connected to vector_database it has animals_text as table
db.open_table("animals_text").head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Vector search in LanceDB

Uses ANN - approximate nearest neighbours for searching vectors

- search with vector directly
- note that the vector column is already an example embedding simulated with LLM

A more realistic approach is to just have text records, use an existing embedding model and embedd those text records, then embed the query vector and find closest match 


In [18]:
db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [19]:
query_vector = [0.5, 0.2, 0.9]

# finds closest rows based on the Euclidean distance by default
db["animals_text"].search(query=query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413


## Embeddings API

LanceDB provides an Embeddings API that allows you to easily generate and store vector embeddings for your data. This is particularly useful for applications such as semantic search, recommendation systems, and clustering.

We'll use embeddings model from sentence-transformer here

In [20]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry
from dotenv import load_dotenv 

load_dotenv()

# NOTE: in .env the key should be called: COHERE_API_KEY
model = get_registry().get("cohere").create(name="embed-multilingual-v3.0")

class JokeModel(LanceModel):
    jokes: str = model.SourceField() 
    vector: Vector(model.ndims()) = model.VectorField()  

print(f"{model.ndims() = }")

db.create_table("jokes", schema=JokeModel, exist_ok=True)

model.ndims() = 1024


LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\indir\\Documents\\github\\pydantic_nvdia\\lancedb\\knowledge_base'))

In [21]:
df = pd.read_json("data/jokes.json")
df.head(2)

,jokes
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."


In [23]:
db["jokes"].add(df)

AddResult(version=2)

In [24]:
db["jokes"].to_pandas().head()

,jokes,vector
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.01239, 0.0047, -0.091431, 0.080627, -0.0505..."
2,What do you call a snake that runs your script...,"[0.019012, 0.044525, -0.021454, 0.02887, -0.00..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019958, 0.060089, 0.012733, 0.024048, -0.07..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013275, 0.002462, -0.057007, 0.060883, -0.0..."


In [25]:
db["jokes"].search("ge mig kemiskämt").limit(4).to_pandas()

,jokes,vector,_distance
0,Oxygen and Magnesium got together—OMg!,"[-0.041138, -0.004559, 0.007397, 0.006527, -0....",1.149439
1,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",1.175357
2,Chemists have all the solutions… mostly in bea...,"[0.00139, 0.030945, -0.001131, 0.021011, -0.05...",1.231428
3,Never trust an atom—they make up everything.,"[-0.009163, 0.003523, -0.020401, 0.049408, -0....",1.269024


In [26]:
df.iloc[0].jokes

'Parallel lines have so much in common—it’s sad they’ll never meet.'

In [27]:
db["jokes"].search(df.iloc[0].jokes).limit(4).to_pandas()

,jokes,vector,_distance
0,Parallel lines have so much in common—it’s sad...,"[-0.00301, 0.001863, -0.014427, 0.051575, -0.0...",0.192628
1,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.188258
2,I tried to explain async/await to my friend… n...,"[0.002825, 0.007874, -0.016235, 0.005787, -0.0...",1.231582
3,My Python code works on the first try… said no...,"[0.020096, 0.035461, -0.008324, 0.048584, -0.0...",1.275363


In [28]:
db["jokes"].search("varför är matteboken ledsen?").limit(4).to_pandas()

,jokes,vector,_distance
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",1.098703
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",1.146541
2,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.152071
3,Why do data engineers hate nature? Too many un...,"[0.038971, 0.061859, -0.02388, 0.050293, -0.03...",1.287978


## Hybrid search

hybrid search combines traditional keyword-based search with vector-based search to provide more relevant results. This approach leverages the strengths of both methods, allowing for more accurate and context-aware retrieval of information.

In [37]:
from lancedb.index import FTS
# keyword based matching on jokes column
db["jokes"].create_index("jokes", config=FTS(),replace=True)

In [38]:
db["jokes"].search(
    "Give me math jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="jokes",
).limit(5).to_pandas()


,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
2,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625


In [39]:
db["jokes"].search(
    "give me math jokes",
    query_type="fts",
    vector_column_name="vector",
    fts_columns="jokes",
).limit(5).to_pandas()

,jokes,vector,_score
0,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",3.530949
1,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",3.324148


In [40]:
db["jokes"].search(
    "give me math jokes",
    query_type="vector",
    vector_column_name="vector",
    fts_columns="jokes",
).limit(5).to_pandas()

,jokes,vector,_distance
0,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.860343
1,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.932817
2,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",1.051328
3,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",1.078976
4,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",1.125342


In [41]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

db["jokes"].search(
    "give me math jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="jokes",
).rerank(reranker).limit(5).to_pandas()

,jokes,vector,_relevance_score
0,My math teacher said I’m average… how mean!,"[-0.002581, 0.031525, 0.013107, 0.030991, -0.0...",0.032258
1,I told a chemistry joke… there was no reaction.,"[-0.032745, 0.018295, 0.039368, 0.001193, -0.0...",0.031778
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005367, 0.020432, -0.002979, 0.016785, -0.0...",0.016393
3,The C# compiler walked into a bar. The bartend...,"[0.003672, 0.006752, -0.076904, 0.026688, -0.0...",0.015873
4,"Dear algebra, please stop asking us to find yo...","[0.011345, 0.004944, -0.019363, 0.060486, -0.0...",0.015625


## Rule of thumb 

- For exact matching -> FTS
- For meaning based matching -> Vector search
- Unpredictable or mixed queries -> Hybrid search
